# Embedding MiniLM + KNN (cosine)
        **text › classification** · notebook `minilm_knn`

        **Kapan dipakai:** Klasifikasi berbasis kemiripan makna; juga alat bantu menemukan contoh mirip.

        **Jebakan umum:** Lambat saat prediksi untuk data besar.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

> Bobot dibaca dari `pretrained/minilm/` (offline). Jika belum ada: `python tools/download_pretrained.py minilm`.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/classification/emot_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/classification/emot_test.csv"
TEXT_COL   = "text"
TARGET     = "label"
ID_COL     = 'id'
LANG       = "id"
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "minilm_knn"
OUT = out_dir("text", "classification", MODEL_KEY)
set_seed(SEED)
MAX_ROWS, MAX_TEST = 3000, 2000        # batasi agar encoding cepat di CPU; naikkan saat lomba

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH) if TEST_PATH else None
if len(df) > MAX_ROWS:
    df = df.groupby(TARGET, group_keys=False).apply(lambda g: g.sample(max(1, int(MAX_ROWS * len(g) / len(df))), random_state=SEED)).reset_index(drop=True)
print(df.shape); display(df[TARGET].value_counts(normalize=True).round(3).to_frame("proporsi"))

## 3. Embedding kalimat (sekali saja, di-cache)
Tiap teks diubah jadi vektor 384 dimensi yang menangkap makna. Embedding bersifat tetap (tidak dilatih), jadi aman dihitung sebelum split dan di-reuse di CV.

In [ ]:
from superdb.text_heavy import SentenceEmbedder
from superdb.text import clean_text
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.tabular import cv_score
texts = df[TEXT_COL].map(clean_text).tolist()
E = SentenceEmbedder("minilm").encode(texts)
le = LabelEncoder(); y = le.fit_transform(df[TARGET]); CLASSES = list(le.classes_)
Etr, Eva, ytr, yva, itr, iva = train_test_split(E, y, np.arange(len(y)), test_size=VALID_SIZE, random_state=SEED, stratify=y)
print(E.shape, "| kelas:", dict(enumerate(CLASSES)))

## 4. Classifier di atas embedding

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=15, metric='cosine', weights='distance')
model

## 5. Cross-validation & evaluasi

In [ ]:
folds, summary = cv_score(model, Etr, ytr, "classification", SCORING, N_SPLITS, SEED); print(summary)
model.fit(Etr, ytr); pred = model.predict(Eva)
proba = model.predict_proba(Eva) if hasattr(model, "predict_proba") else None
metrics = classification_metrics(yva, pred, proba, classes=np.arange(len(CLASSES))); display(show_table(metrics))
from sklearn.metrics import classification_report
print(classification_report(yva, pred, target_names=[str(c) for c in CLASSES]))
plot_confusion(yva, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()

## 6. Bandingkan dengan baseline TF-IDF (data yang sama)
Pembanding wajib: embedding belum tentu menang.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
tf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(C=5, max_iter=2000))
tf.fit([texts[i] for i in itr], ytr)
f1_tf = f1_score(yva, tf.predict([texts[i] for i in iva]), average="macro")
print(f"F1 macro validation | TF-IDF+LogReg: {f1_tf:.4f} | embedding+{MODEL_KEY.split('minilm_')[1]}: {metrics['f1_macro']:.4f}")
metrics["f1_macro_tfidf_baseline"] = float(f1_tf)

## 7. Latih ulang pada semua data → submission

In [ ]:
from sklearn.base import clone
final = clone(model).fit(E, y)
if test is not None:
    t = test.head(MAX_TEST); Et = SentenceEmbedder("minilm").encode(t[TEXT_COL].map(clean_text).tolist())
    sub = make_submission(t[ID_COL] if ID_COL else np.arange(len(t)), le.inverse_transform(final.predict(Et)), ID_COL or "id", TARGET, OUT / "submission.csv"); display(sub.head(3))
save_model({"clf": final, "classes": CLASSES, "embedder": "minilm"}, OUT / "model.joblib")
save_json({"cv": summary, "valid": metrics, "model": MODEL_KEY}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib"); assert (b["clf"].predict(E[:50]) == final.predict(E[:50])).all(); print("OK. hasil di", OUT)

## Langkah berikutnya
- Gabungkan prediksi embedding dengan TF-IDF (blend): kesalahannya biasanya berbeda.
- Data berbahasa Indonesia dengan banyak slang: coba fine-tuning (`finetune_*`).